# Lineborn Sales LoRA v3
Pinned September 2026 training stack, full subprocess logs, and fail-closed adapter packaging. Use a fresh GPU runtime and Run all.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

def run(*args):
    cmd = list(map(str, args))
    print('>', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='', flush=True)
        tail.append(line)
        if len(tail) > 120:
            tail.pop(0)
    code = proc.wait()
    if code:
        raise RuntimeError(f'Command failed with exit code {code}: {cmd}\n\nLAST OUTPUT:\n' + ''.join(tail))
    return code

run('nvidia-smi')
print('Notebook Python:', sys.version)
print('Executable:', sys.executable)
root = Path('/content/lineborn-runtime')
if root.exists():
    shutil.rmtree(root)
run('git', 'clone', '--depth', '1', '--branch', 'lineborn-sales-lora', 'https://github.com/SumamaAhmed69/Axemetric-Caller-Beta-Runtime.git', str(root))
os.chdir(root)
run('git', 'rev-parse', 'HEAD')
run(sys.executable, '-m', 'pip', 'install', '--upgrade', '-q', '-r', 'training/requirements-colab.txt')
run(sys.executable, '-m', 'pip', 'check')
run(sys.executable, '-c', "import sys,torch,transformers,peft,trl,datasets,accelerate,bitsandbytes; print('python',sys.version); print('torch',torch.__version__,'cuda',torch.version.cuda,'gpu',torch.cuda.get_device_name(0) if torch.cuda.is_available() else None); print('transformers',transformers.__version__); print('peft',peft.__version__); print('trl',trl.__version__); print('datasets',datasets.__version__); print('accelerate',accelerate.__version__); print('bitsandbytes',bitsandbytes.__version__)")
print('Environment ready:', root)

In [ ]:
run(sys.executable, 'training/build_sales_corpus.py')
run(sys.executable, 'training/validate_sales_corpus.py')

In [ ]:
run(sys.executable, 'training/train_sales_lora.py', '--max-length', '1536', '--epochs', '2', '--grad-accum', '16')
sft_model = Path('training/output/lineborn-sales-sft/adapter/adapter_model.safetensors')
assert sft_model.is_file(), 'SFT adapter_model.safetensors missing'
assert sft_model.stat().st_size > 1024 * 1024, f'SFT adapter suspiciously small: {sft_model.stat().st_size} bytes'
print(f'SFT adapter ready: {sft_model.stat().st_size / 1024 / 1024:.1f} MiB')

In [ ]:
run(sys.executable, 'training/train_sales_dpo.py', '--max-length', '1536', '--epochs', '1', '--grad-accum', '16')
dpo_model = Path('training/output/lineborn-sales-dpo/adapter/adapter_model.safetensors')
assert dpo_model.is_file(), 'DPO adapter_model.safetensors missing'
assert dpo_model.stat().st_size > 1024 * 1024, f'DPO adapter suspiciously small: {dpo_model.stat().st_size} bytes'
print(f'DPO adapter ready: {dpo_model.stat().st_size / 1024 / 1024:.1f} MiB')

In [ ]:
run(sys.executable, 'training/package_sales_adapters.py', '--require-dpo', '--archive', '/content/lineborn-sales-adapters-v3.zip')
archive = Path('/content/lineborn-sales-adapters-v3.zip')
assert archive.is_file(), 'candidate archive missing'
assert archive.stat().st_size > 2 * 1024 * 1024, f'candidate archive suspiciously small: {archive.stat().st_size} bytes'
print(f'Candidate archive ready: {archive.stat().st_size / 1024 / 1024:.1f} MiB')
from google.colab import files
files.download(str(archive))